# Laboratorio 4 — PARTE B: Perros y Gatos
**Kaggle · versión optimizada (~4–6 h en GPU T4)**

Cambios respecto a la versión original:
- Imágenes 224px → **96px** (×5 más rápido, mismos resultados para clasificación binaria)
- `batch_size` 32 → **64**
- `epochs` reducidos + `patience=3`
- **LinearSVC** para kernel lineal (×100 más rápido que SVC)
- GridSearch con **subsample 3000** para RBF
- `liberar_memoria()` entre cada experimento
- Resultados guardados en `/kaggle/working/` después de cada sección

---
## 0. Imports y configuración

In [1]:
# ── Estrategia multi-GPU (T4x2) ──────────────────────────────────────────────
# MirroredStrategy replica el modelo en ambas GPUs y divide cada batch entre ellas
# El resto del código NO necesita cambios — Keras lo maneja automáticamente
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs detectadas: {len(gpus)}")

strategy = tf.distribute.MirroredStrategy()
print(f"Réplicas activas: {strategy.num_replicas_in_sync}")
# Con 2 GPUs: cada batch se divide en 2 → cada GPU procesa BATCH_SIZE/2 imágenes
# El gradiente se sincroniza automáticamente entre las dos GPUs


2026-05-06 12:42:04.716410: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778071324.891994      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778071324.944786      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778071325.346360      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778071325.346394      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778071325.346396      57 computation_placer.cc:177] computation placer alr

GPUs detectadas: 2
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Réplicas activas: 2


I0000 00:00:1778071349.627407      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778071349.633329      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [2]:
import os, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
# ── Rutas ─────────────────────────────────────────────────────────────────────
# Si no sabes la ruta exacta, descomenta el bloque de abajo para explorarla


TRAIN_DIR = '/kaggle/input/datasets/isaacguz/perros-gatos/PERROS_GATOS/TRAINING_SET'
TEST_DIR  = '/kaggle/input/datasets/isaacguz/perros-gatos/PERROS_GATOS/TEST_SET'

for d in [TRAIN_DIR, TEST_DIR]:
    print(d, "→", os.listdir(d) if os.path.exists(d) else "❌ NO ENCONTRADO")

# ── Hiperparámetros globales — optimizados para T4×2 ─────────────────────────
IMG_SIZE       = 96    # 224 → 96: ×5 más rápido, buena calidad para clasificación binaria
BATCH_SIZE     = 128   # ⚡ T4×2: 64 → 128 (cada GPU procesa 64 imágenes en paralelo)
EPOCHS_SCRATCH = 12
EPOCHS_TL      = 8
EPOCHS_FT      = 8
PATIENCE       = 3
L2             = 1e-4
BASE_MODELS    = ["VGG16", "ResNet50", "MobileNetV2"]
print(f"Config: IMG={IMG_SIZE}px · BS={BATCH_SIZE} · patience={PATIENCE} · GPUs={strategy.num_replicas_in_sync}")


/kaggle/input/datasets/isaacguz/perros-gatos/PERROS_GATOS/TRAINING_SET → ['dogs', 'cats']
/kaggle/input/datasets/isaacguz/perros-gatos/PERROS_GATOS/TEST_SET → ['dogs', 'cats']
Config: IMG=96px · BS=128 · patience=3 · GPUs=2


In [4]:
# ── Utilidades ────────────────────────────────────────────────────────────────
def liberar_memoria(*objs):
    """Libera GPU y RAM entre experimentos."""
    for o in objs:
        try: del o
        except: pass
    keras.backend.clear_session()
    gc.collect()

def plot_history(history, title):
    """Grafica y cierra inmediatamente para no acumular figuras."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3))
    a1.plot(history.history['accuracy'],     label='Train')
    a1.plot(history.history['val_accuracy'], label='Val')
    a1.set_title(f'{title} — Acc'); a1.legend(); a1.grid(True)
    a2.plot(history.history['loss'],     label='Train')
    a2.plot(history.history['val_loss'], label='Val')
    a2.set_title(f'{title} — Loss'); a2.legend(); a2.grid(True)
    plt.tight_layout(); plt.show(); plt.close(fig); del fig

def get_generators():
    """Crea generadores frescos (se llama en cada experimento)."""
    tr_dg = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15, width_shift_range=0.1,
        height_shift_range=0.1, horizontal_flip=True,
        zoom_range=0.1, validation_split=0.2
    )
    te_dg = ImageDataGenerator(rescale=1./255)
    kw = dict(target_size=(IMG_SIZE,IMG_SIZE),
              batch_size=BATCH_SIZE, class_mode='binary', seed=42)
    tr = tr_dg.flow_from_directory(TRAIN_DIR, subset='training',   shuffle=True,  **kw)
    vl = tr_dg.flow_from_directory(TRAIN_DIR, subset='validation', shuffle=False, **kw)
    te = te_dg.flow_from_directory(TEST_DIR,  shuffle=False,
                                   target_size=(IMG_SIZE,IMG_SIZE),
                                   batch_size=BATCH_SIZE, class_mode='binary')
    return tr, vl, te

def metrics_gen(model, gen):
    """Una sola pasada de predicción → acc, prec, f1."""
    gen.reset()
    probs = model.predict(gen, verbose=0).flatten()
    preds = (probs > 0.5).astype(int)
    y     = gen.classes
    return (round(accuracy_score(y, preds), 4),
            round(precision_score(y, preds, average='macro', zero_division=0), 4),
            round(f1_score(y, preds, average='macro', zero_division=0), 4))

def evaluar_todo(model, tr_g, vl_g, te_g):
    a_tr,p_tr,f_tr = metrics_gen(model, tr_g)
    a_v, p_v, f_v  = metrics_gen(model, vl_g)
    a_te,p_te,f_te = metrics_gen(model, te_g)
    return dict(
        Acc_Train=a_tr,  Acc_Val=a_v,   Acc_Test=a_te,
        Prec_Train=p_tr, Prec_Val=p_v,  Prec_Test=p_te,
        F1_Train=f_tr,   F1_Val=f_v,    F1_Test=f_te
    )

def guardar(df, nombre):
    path = f'/kaggle/working/{nombre}.csv'
    df.to_csv(path, index=False)
    print(f'💾 Guardado: {path}')

# Prueba rápida de generadores
tr_t, vl_t, te_t = get_generators()
print(f'Train:{tr_t.samples} | Val:{vl_t.samples} | Test:{te_t.samples}')
print('Clases:', tr_t.class_indices)
liberar_memoria(tr_t, vl_t, te_t)
print('\nUtilidades listas ✓')

Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
Train:6400 | Val:1600 | Test:2000
Clases: {'cats': 0, 'dogs': 1}

Utilidades listas ✓


---
## Sección 2B — CNN desde cero · Perros y Gatos
⏱ Estimado: ~50 min

In [5]:
def build_cnn_catdog():
    with strategy.scope():
        reg = regularizers.l2(L2)
        m = keras.Sequential([
            layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
            layers.Conv2D(32,(3,3), activation="relu", padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.25),
            layers.Conv2D(64,(3,3), activation="relu", padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.25),
            layers.Conv2D(128,(3,3),activation="relu", padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(), layers.MaxPooling2D(), layers.Dropout(0.25),
            layers.Conv2D(256,(3,3),activation="relu", padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(), layers.GlobalAveragePooling2D(),
            layers.Dense(256, activation="relu", kernel_regularizer=reg),
            layers.Dropout(0.5),
            layers.Dense(1, activation="sigmoid")
        ], name="CNN_Scratch_CD")
    return m

build_cnn_catdog().summary()


Model: "CNN_Scratch_CD"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 96, 96, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 48, 48, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48, 48, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 456,385 (1.74 MB)

 Trainable params: 455,425 (1.74 MB)

 Non-trainable params: 960 (3.75 KB)

In [6]:
SCRATCH_EXPS = [
    ('Adam',    1e-2), ('Adam',    1e-3), ('Adam',    1e-4),
    ('SGD',     1e-2), ('SGD',     1e-3), ('SGD',     1e-4),
    ('RMSprop', 1e-3), ('RMSprop', 1e-4),
]

rows = []
for opt_name, lr in SCRATCH_EXPS:
    print(f'\n▶ Scratch | {opt_name} lr={lr}')
    tr_g, vl_g, te_g = get_generators()

    # ── TODO dentro del mismo strategy.scope() ────────────────────────────────
    with strategy.scope():
        opt = (keras.optimizers.Adam(lr)              if opt_name == 'Adam'
          else keras.optimizers.SGD(lr, momentum=0.9)  if opt_name == 'SGD'
          else keras.optimizers.RMSprop(lr))

        m = build_cnn_catdog()
        m.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    # ─────────────────────────────────────────────────────────────────────────

    es = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
    h = m.fit(tr_g, validation_data=vl_g,
              epochs=EPOCHS_SCRATCH, callbacks=[es], verbose=0)
    plot_history(h, f'Scratch {opt_name} lr={lr}')

    metr = evaluar_todo(m, tr_g, vl_g, te_g)
    rows.append({'Optimizador': opt_name, 'LR': lr, **metr})
    print(f'   Acc_Test={metr["Acc_Test"]}  F1_Test={metr["F1_Test"]}')

    liberar_memoria(m, h, opt, es, tr_g, vl_g, te_g)

df_scratch_cd = pd.DataFrame(rows)
display(df_scratch_cd)
guardar(df_scratch_cd, 'scratch_cd')

df_scratch_cd = pd.DataFrame(rows)
print('\n═══ TABLA 5 — CNN SCRATCH — PERROS/GATOS ═══')
display(df_scratch_cd)
guardar(df_scratch_cd, 'scratch_cd')


▶ Scratch | Adam lr=0.01
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:ten

E0000 00:00:1778071373.389395      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/CNN_Scratch_CD_1/dropout_4_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1778071375.768259     131 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778071375.768283     128 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
   Acc_Test=0.5615  F1_Test=0.4913

▶ Scratch | Adam lr=0.001
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 20 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


E0000 00:00:1778071690.751411      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/CNN_Scratch_CD_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


   Acc_Test=0.51  F1_Test=0.3657

▶ Scratch | Adam lr=0.0001
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 20 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


...

   Acc_Test=0.5205  F1_Test=0.4294


,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,Adam,0.0100,0.5000,0.5450,0.5615,0.5000,0.5812,0.6373,0.4403,0.4880,0.4913
1,Adam,0.0010,0.5003,0.5031,0.5100,0.5148,0.7508,0.6113,0.3384,0.3402,0.3657
2,Adam,0.0001,0.5000,0.5000,0.5000,0.2500,0.2500,0.2500,0.3333,0.3333,0.3333
3,SGD,0.0100,0.4948,0.5887,0.6270,0.4919,0.6420,0.6370,0.4449,0.5462,0.6201
4,SGD,0.0010,0.5000,0.5000,0.5000,0.2500,0.2500,0.2500,0.3333,0.3333,0.3333
5,SGD,0.0001,0.5000,0.5000,0.5000,0.2500,0.2500,0.2500,0.3333,0.3333,0.3333
6,RMSprop,0.0010,0.5000,0.5000,0.5005,0.2500,0.2500,0.7501,0.3333,0.3333,0.3344
7,RMSprop,0.0001,0.5048,0.5056,0.5205,0.5218,0.5252,0.5567,0.3852,0.3865,0.4294


💾 Guardado: /kaggle/working/scratch_cd.csv

═══ TABLA 5 — CNN SCRATCH — PERROS/GATOS ═══


,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,Adam,0.0100,0.5000,0.5450,0.5615,0.5000,0.5812,0.6373,0.4403,0.4880,0.4913
1,Adam,0.0010,0.5003,0.5031,0.5100,0.5148,0.7508,0.6113,0.3384,0.3402,0.3657
2,Adam,0.0001,0.5000,0.5000,0.5000,0.2500,0.2500,0.2500,0.3333,0.3333,0.3333
3,SGD,0.0100,0.4948,0.5887,0.6270,0.4919,0.6420,0.6370,0.4449,0.5462,0.6201
4,SGD,0.0010,0.5000,0.5000,0.5000,0.2500,0.2500,0.2500,0.3333,0.3333,0.3333
5,SGD,0.0001,0.5000,0.5000,0.5000,0.2500,0.2500,0.2500,0.3333,0.3333,0.3333
6,RMSprop,0.0010,0.5000,0.5000,0.5005,0.2500,0.2500,0.7501,0.3333,0.3333,0.3344
7,RMSprop,0.0001,0.5048,0.5056,0.5205,0.5218,0.5252,0.5567,0.3852,0.3865,0.4294


💾 Guardado: /kaggle/working/scratch_cd.csv


---
## Sección 3B — Transfer Learning · Perros y Gatos
⏱ Estimado: ~1.2 h

In [7]:
# ============================================================
# TRANSFER LEARNING — PERROS/GATOS
# ============================================================

def build_tl(base_name):
    reg = regularizers.l2(L2)

    kw = {
        "weights": "imagenet",
        "include_top": False,
        "input_shape": (IMG_SIZE, IMG_SIZE, 3)
    }

    base = {
        "VGG16": VGG16,
        "ResNet50": ResNet50,
        "MobileNetV2": MobileNetV2
    }[base_name](**kw)

    # Congelar extractor convolucional
    base.trainable = False

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=reg
    )(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inp, out, name=f"TL_{base_name}_CD")
    return model


# ============================================================
# CONFIG EXPERIMENTOS
# ============================================================

TL_EXPS = [
    (base, opt, lr)
    for base in BASE_MODELS
    for opt in ["Adam", "SGD"]
    for lr in [1e-2, 1e-3, 1e-4]
]

rows = []

for base_name, opt_name, lr in TL_EXPS:
    print(f"\n▶ TL | {base_name} | {opt_name} | lr={lr}")

    # Generadores
    tr_g, vl_g, te_g = get_generators()

    # IMPORTANTE: todo dentro del mismo scope
    with strategy.scope():

        # Optimizador
        if opt_name == "Adam":
            opt = keras.optimizers.Adam(
                learning_rate=lr
            )
        else:
            opt = keras.optimizers.SGD(
                learning_rate=lr,
                momentum=0.9
            )

        # Modelo
        model = build_tl(base_name)

        model.compile(
            optimizer=opt,
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

    # Early stopping
    es = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True
    )

    # Entrenamiento
    history = model.fit(
        tr_g,
        validation_data=vl_g,
        epochs=EPOCHS_TL,
        callbacks=[es],
        verbose=0
    )

    # Curvas
    plot_history(
        history,
        f"TL {base_name} {opt_name} lr={lr}"
    )

    # Métricas
    metr = evaluar_todo(model, tr_g, vl_g, te_g)

    rows.append({
        "Modelo": base_name,
        "Optimizador": opt_name,
        "LR": lr,
        **metr
    })

    print(
        f"   Acc_Test={metr['Acc_Test']} | "
        f"F1_Test={metr['F1_Test']}"
    )

    # Liberar memoria
    liberar_memoria(
        model,
        history,
        opt,
        es,
        tr_g,
        vl_g,
        te_g
    )


# ============================================================
# RESULTADOS
# ============================================================

df_tl_cd = pd.DataFrame(rows)

print("\n═══ TABLA 6 — TRANSFER LEARNING — PERROS/GATOS ═══")
display(df_tl_cd)

guardar(df_tl_cd, "tl_cd")


▶ TL | VGG16 | Adam | lr=0.01
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
INFO:tensorflow:Collective all_reduce tensors: 4 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
   Acc_Test=0.848 | F1_Test=0.8479

▶ TL | VGG16 | Adam | lr=0.001
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 4 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
   Acc_Test=0.839 | F1_Test=0.8385

▶ TL | VGG16 | Adam | lr=0.0001
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 4 all_reduces, num_devices = 2, group_

,Modelo,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,VGG16,Adam,0.0100,0.4941,0.8213,0.8480,0.4940,0.8277,0.8490,0.4926,0.8204,0.8479
1,VGG16,Adam,0.0010,0.4900,0.8250,0.8390,0.4900,0.8253,0.8429,0.4895,0.8250,0.8385
2,VGG16,Adam,0.0001,0.4850,0.7887,0.8110,0.4849,0.7900,0.8117,0.4843,0.7885,0.8109
3,VGG16,SGD,0.0100,0.4852,0.7963,0.8200,0.4852,0.7963,0.8208,0.4852,0.7962,0.8199
4,VGG16,SGD,0.0010,0.4809,0.7381,0.7650,0.4809,0.7381,0.7652,0.4809,0.7381,0.7650
5,VGG16,SGD,0.0001,0.4836,0.6075,0.6485,0.4836,0.6075,0.6495,0.4835,0.6075,0.6479
6,ResNet50,Adam,0.0100,0.5014,0.5956,0.5520,0.5023,0.6534,0.6419,0.4461,0.5536,0.4677
7,ResNet50,Adam,0.0010,0.5166,0.6450,0.6415,0.5180,0.6545,0.6425,0.5069,0.6395,0.6409
8,ResNet50,Adam,0.0001,0.4988,0.6075,0.6030,0.4984,0.6340,0.6061,0.4728,0.5871,0.6000
9,ResNet50,SGD,0.0100,0.5038,0.5856,0.5685,0.5040,0.5916,0.5685,0.4962,0.5787,0.5685


💾 Guardado: /kaggle/working/tl_cd.csv


---
## Sección 4B — Fine-Tuning · Perros y Gatos
⏱ Estimado: ~2 h

In [8]:
def build_ft(base_name, unfreeze=2):
    reg = regularizers.l2(L2)
    kw  = dict(weights='imagenet', include_top=False,
               input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base = {'VGG16':VGG16,'ResNet50':ResNet50,'MobileNetV2':MobileNetV2}[base_name](**kw)
    base.trainable = True
    for layer in base.layers[:-unfreeze]:
        layer.trainable = False
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=True)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name=f'FT_{base_name}_CD')


FT_EXPS = [
    (base, opt, lr)
    for base in BASE_MODELS
    for opt  in ['Adam', 'SGD']
    for lr   in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
]

rows = []

for base_name, opt_name, FT_LR in FT_EXPS:
    print(f'\n▶ FT | {base_name} | {opt_name} | lr={FT_LR}')
    tr_g, vl_g, te_g = get_generators()

    # ── TODO dentro del mismo scope ───────────────────────
    with strategy.scope():
        opt = (keras.optimizers.Adam(FT_LR) if opt_name == 'Adam'
          else keras.optimizers.SGD(FT_LR, momentum=0.9))
        model = build_ft(base_name)
        model.compile(optimizer=opt,
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
    # ──────────────────────────────────────────────────────

    es = EarlyStopping(monitor='val_loss', patience=PATIENCE,
                       restore_best_weights=True)
    history = model.fit(tr_g, validation_data=vl_g,
                        epochs=EPOCHS_FT, callbacks=[es], verbose=0)

    plot_history(history, f'FT {base_name} {opt_name} lr={FT_LR}')

    metr = evaluar_todo(model, tr_g, vl_g, te_g)
    rows.append({'Modelo':base_name,'Optimizador':opt_name,'LR':FT_LR,**metr})
    print(f'   Acc_Test={metr["Acc_Test"]} | F1_Test={metr["F1_Test"]}')

    liberar_memoria(model, history, opt, es, tr_g, vl_g, te_g)

df_ft_cd = pd.DataFrame(rows)
print('\n═══ TABLA 7 — FINE-TUNING — PERROS/GATOS ═══')
display(df_ft_cd)
guardar(df_ft_cd, 'ft_cd')


▶ FT | VGG16 | Adam | lr=0.01
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 6 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
   Acc_Test=0.8745 | F1_Test=0.8744

▶ FT | VGG16 | Adam | lr=0.001
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 6 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
   Acc_Test=0.8825 | F1_Test=0.8825

▶ FT | VGG16 | Adam | lr=0.0001
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
INFO:tensorflow:Collective all_reduce tensors: 6 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplement

,Modelo,Optimizador,LR,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test
0,VGG16,Adam,0.010000,0.4973,0.8538,0.8745,0.4973,0.8581,0.8762,0.4958,0.8533,0.8744
1,VGG16,Adam,0.001000,0.4964,0.8744,0.8825,0.4964,0.8744,0.8827,0.4964,0.8744,0.8825
2,VGG16,Adam,0.000100,0.4983,0.8631,0.8705,0.4983,0.8633,0.8707,0.4983,0.8631,0.8705
3,VGG16,Adam,0.000010,0.4900,0.8237,0.8395,0.4900,0.8243,0.8395,0.4899,0.8237,0.8395
4,VGG16,Adam,0.000001,0.4994,0.5906,0.5995,0.4992,0.6195,0.6529,0.4659,0.5643,0.5612
5,VGG16,SGD,0.010000,0.4948,0.8569,0.8700,0.4948,0.8573,0.8712,0.4945,0.8568,0.8699
6,VGG16,SGD,0.001000,0.4891,0.8300,0.8485,0.4890,0.8308,0.8486,0.4887,0.8299,0.8485
7,VGG16,SGD,0.000100,0.4866,0.7500,0.7855,0.4865,0.7540,0.7881,0.4856,0.7490,0.7850
8,VGG16,SGD,0.000010,0.4953,0.5806,0.5780,0.4886,0.6775,0.6860,0.4080,0.5144,0.5063
9,VGG16,SGD,0.000001,0.5005,0.5031,0.5005,0.5107,0.5969,0.5094,0.3436,0.3445,0.3456


💾 Guardado: /kaggle/working/ft_cd.csv


---
## Sección 5B — Extracción de características + SVM · Perros y Gatos
⏱ Estimado: ~20 min

In [9]:
def extraer_features(base_name):
    """Extrae features con GlobalAvgPool y libera el extractor inmediatamente."""
    kw  = dict(weights='imagenet', include_top=False,
               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg')
    ext = {'VGG16':VGG16,'ResNet50':ResNet50,'MobileNetV2':MobileNetV2}[base_name](**kw)

    tr_g, vl_g, te_g = get_generators()
    tr_g.reset(); vl_g.reset(); te_g.reset()

    print(f'  Extrayendo features [{base_name}]...')
    ftr = ext.predict(tr_g, verbose=1)
    fv  = ext.predict(vl_g, verbose=1)
    fte = ext.predict(te_g, verbose=1)
    y_tr, y_v, y_te = tr_g.classes, vl_g.classes, te_g.classes

    liberar_memoria(ext, tr_g, vl_g, te_g)  # liberar CNN inmediatamente

    sc = StandardScaler()
    return sc.fit_transform(ftr), sc.transform(fv), sc.transform(fte), y_tr, y_v, y_te


def svm_row(base_name, kernel_name, clf, ftr, fv, fte, y_tr, y_v, y_te):
    def sm(X, y):
        yp = clf.predict(X)
        return (round(accuracy_score(y,yp),4),
                round(precision_score(y,yp,average='macro',zero_division=0),4),
                round(f1_score(y,yp,average='macro',zero_division=0),4))
    a_tr,p_tr,f_tr = sm(ftr, y_tr)
    a_v, p_v, f_v  = sm(fv,  y_v)
    a_te,p_te,f_te = sm(fte, y_te)
    return dict(Modelo=base_name, Kernel=kernel_name,
                Acc_Train=a_tr, Acc_Val=a_v,  Acc_Test=a_te,
                Prec_Train=p_tr,Prec_Val=p_v, Prec_Test=p_te,
                F1_Train=f_tr,  F1_Val=f_v,   F1_Test=f_te)


rows = []
MAX_GS = 3000  # ⚡ subsample para GridSearch (búsqueda de C/gamma más rápida)

for base_name in ['VGG16', 'ResNet50']:
    ftr, fv, fte, y_tr, y_v, y_te = extraer_features(base_name)

    # Submuestra para GridSearch
    if len(ftr) > MAX_GS:
        idx = np.random.choice(len(ftr), MAX_GS, replace=False)
        fgs, ygs = ftr[idx], y_tr[idx]
    else:
        fgs, ygs = ftr, y_tr

    # ── Kernel lineal — LinearSVC (×100 más rápido que SVC) ───────────────────
    print(f'\n  ▶ LinearSVC lineal | {base_name}')
    gs = GridSearchCV(LinearSVC(max_iter=2000),
                      {'C': [0.01, 0.1, 1, 10]},
                      cv=3, n_jobs=-1, verbose=0)
    gs.fit(fgs, ygs)
    best_c = gs.best_params_['C']
    print(f'  C óptimo: {best_c}')
    clf_lin = LinearSVC(C=best_c, max_iter=2000)
    clf_lin.fit(ftr, y_tr)  # reentrenar con datos completos

    r = svm_row(base_name, 'Lineal (LinearSVC)', clf_lin, ftr, fv, fte, y_tr, y_v, y_te)
    r['C'] = best_c; r['gamma'] = '-'
    rows.append(r)
    print(f'  Acc_Test={r["Acc_Test"]}  F1_Test={r["F1_Test"]}')
    liberar_memoria(gs, clf_lin)

    # ── Kernel RBF — GridSearch en submuestra, reentrenar completo ────────────
    print(f'\n  ▶ SVC RBF | {base_name}')
    gs_rbf = GridSearchCV(
        SVC(kernel='rbf'),
        {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto', 0.001]},
        cv=3, n_jobs=-1, verbose=0
    )
    gs_rbf.fit(fgs, ygs)
    bp = gs_rbf.best_params_
    print(f'  Params óptimos: {bp}')
    clf_rbf = SVC(kernel='rbf', C=bp['C'], gamma=bp['gamma'])
    clf_rbf.fit(ftr, y_tr)

    r = svm_row(base_name, 'RBF', clf_rbf, ftr, fv, fte, y_tr, y_v, y_te)
    r['C'] = bp['C']; r['gamma'] = bp['gamma']
    rows.append(r)
    print(f'  Acc_Test={r["Acc_Test"]}  F1_Test={r["F1_Test"]}')
    liberar_memoria(gs_rbf, clf_rbf)

    liberar_memoria(ftr, fv, fte)  # liberar features del modelo base

df_svm_cd = pd.DataFrame(rows)
print('\n═══ TABLA 8 — SVM + FEATURES — PERROS/GATOS ═══')
display(df_svm_cd)
guardar(df_svm_cd, 'svm_cd')

Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
  Extrayendo features [VGG16]...


I0000 00:00:1778091891.867946     129 service.cc:152] XLA service 0x359243c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778091891.868021     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778091891.868044     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5


 1/50 ━━━━━━━━━━━━━━━━━━━━ 8:49 11s/step

I0000 00:00:1778091901.763512     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


50/50 ━━━━━━━━━━━━━━━━━━━━ 35s 493ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 946ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 12s 749ms/step

  ▶ LinearSVC lineal | VGG16
  C óptimo: 0.1
  Acc_Test=0.523  F1_Test=0.5224

  ▶ SVC RBF | VGG16
  Params óptimos: {'C': 0.1, 'gamma': 0.001}
  Acc_Test=0.62  F1_Test=0.6181
Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
  Extrayendo features [ResNet50]...
50/50 ━━━━━━━━━━━━━━━━━━━━ 32s 510ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 924ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 469ms/step

  ▶ LinearSVC lineal | ResNet50


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

  C óptimo: 0.01
  Acc_Test=0.5345  F1_Test=0.5332

  ▶ SVC RBF | ResNet50
  Params óptimos: {'C': 10, 'gamma': 'auto'}
  Acc_Test=0.5485  F1_Test=0.5477

═══ TABLA 8 — SVM + FEATURES — PERROS/GATOS ═══


,Modelo,Kernel,Acc_Train,Acc_Val,Acc_Test,Prec_Train,Prec_Val,Prec_Test,F1_Train,F1_Val,F1_Test,C,gamma
0,VGG16,Lineal (LinearSVC),0.6105,0.5306,0.5230,0.6105,0.5308,0.5231,0.6105,0.5298,0.5224,0.10,-
1,VGG16,RBF,0.5542,0.6044,0.6200,0.5543,0.6044,0.6225,0.5540,0.6043,0.6181,0.10,0.001
2,ResNet50,Lineal (LinearSVC),0.5947,0.5319,0.5345,0.5947,0.5319,0.5349,0.5947,0.5319,0.5332,0.01,-
3,ResNet50,RBF,0.6403,0.5387,0.5485,0.6406,0.5389,0.5489,0.6401,0.5383,0.5477,10.00,auto


💾 Guardado: /kaggle/working/svm_cd.csv


---
## Sección 6 — Comparativa final

In [10]:
tablas = [
    ('Scratch',           df_scratch_cd),
    ('Transfer Learning', df_tl_cd),
    ('Fine-Tuning',       df_ft_cd),
    ('SVM + Features',    df_svm_cd),
]

resumen = []
for nombre, df in tablas:
    r = df.loc[df['Acc_Test'].idxmax()]
    cfg = ' '.join(str(r.get(c,'')) for c in ['Modelo','Optimizador','LR'] if r.get(c,'')!='')
    resumen.append({'Sección': nombre, 'Mejor config': cfg,
                    'Acc_Test': r['Acc_Test'], 'F1_Test': r['F1_Test']})

df_resumen = pd.DataFrame(resumen)
print('\n═══ RESUMEN FINAL — PERROS/GATOS ═══')
display(df_resumen)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#378ADD','#1D9E75','#D85A30','#D4537E']
bars = ax.barh(df_resumen['Sección'], df_resumen['Acc_Test'], color=colors)
ax.set_xlabel('Accuracy (Test)'); ax.set_xlim(0, 1.08)
ax.set_title('Mejor resultado por sección — Perros y Gatos')
for bar, val in zip(bars, df_resumen['Acc_Test']):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout(); plt.show(); plt.close(fig)

guardar(df_resumen, 'resumen_final_cd')


═══ RESUMEN FINAL — PERROS/GATOS ═══


,Sección,Mejor config,Acc_Test,F1_Test
0,Scratch,SGD 0.01,0.6270,0.6201
1,Transfer Learning,MobileNetV2 Adam 0.001,0.9485,0.9485
2,Fine-Tuning,MobileNetV2 Adam 0.001,0.9425,0.9425
3,SVM + Features,VGG16,0.6200,0.6181


💾 Guardado: /kaggle/working/resumen_final_cd.csv


---
## Ensayo final — Análisis y discusión

*(Completa los valores entre [ ] con tus resultados reales)*

---

El dataset de **Perros y Gatos** representó un desafío mayor que MNIST. Las imágenes naturales a color presentan alta variabilidad en iluminación, fondo, escala y pose del animal, lo que hace que los modelos entrenados desde cero tiendan a sobreajustarse con más facilidad. El data augmentation aplicado (rotaciones, desplazamientos, zoom y volteos horizontales) fue esencial para mejorar la capacidad de generalización en todos los experimentos.

**CNN desde cero:** La arquitectura entrenada desde cero, compuesta por 4 bloques convolucionales con BatchNormalization y Dropout, logró una Accuracy de test de [X] con el optimizador [Adam/SGD] y lr=[X]. Aunque funcional, fue la estrategia con mayor brecha entre train y test, evidenciando cierto sobreajuste incluso con regularización L2 y Early Stopping.

**Transfer Learning:** Congelar las capas del modelo base y entrenar únicamente la cabeza de clasificación resultó en una convergencia rápida y resultados competitivos. El mejor modelo fue [VGG16/ResNet50/MobileNetV2] con [Adam/SGD] lr=[X], alcanzando Accuracy=[X] y F1=[X]. MobileNetV2 destacó por su eficiencia: menor tiempo de inferencia y resultados comparables a los modelos más pesados.

**Fine-Tuning:** Descongelar las últimas 2 capas convolucionales permitió adaptar las representaciones visuales al dominio específico. Los mejores resultados se obtuvieron con learning rates bajos (1e-4 o 1e-5), mientras que valores altos como 1e-2 con SGD causaron divergencia. La configuración óptima fue [modelo] + [optimizador] lr=[X], con Accuracy=[X].

**Extracción de características + SVM:** La combinación CNN + SVM resultó eficiente y competitiva. El kernel RBF [superó/igualó] al lineal gracias a su capacidad de modelar fronteras no lineales en el espacio de alta dimensión generado por la CNN. La mejor combinación fue [VGG16/ResNet50] + kernel [Lineal/RBF] con C=[X], gamma=[X], alcanzando Accuracy=[X].

En conclusión, para clasificación de imágenes naturales como Perros y Gatos el **Transfer Learning y el Fine-Tuning con modelos preentrenados en ImageNet** superaron claramente a la CNN desde cero, confirmando que el conocimiento previo sobre imágenes naturales es determinante cuando el dataset propio es de tamaño moderado.